In [1]:
import glob
import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil
import torch
import torchvision.transforms as T
import random
from torch.utils.data import Dataset, DataLoader
import torchvision.utils as vutils
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models

In [2]:
#Use cuda if it is available in device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
# Create custom pytorch Dataset class whcih reads the image from dataframe
class HandWrittenImageDataset(Dataset):
    def __init__(self, dataframe, transform=None, num_images=None):
        self.dataset = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img_path = self.dataset.iloc[idx]["Images"]
#         one_hot_label = self.dataset.iloc[idx]["one_hot_encoded_labels"]
        one_hot_label = torch.tensor(self.dataset.iloc[idx]["one_hot_encoded_labels"])
        label = self.dataset.iloc[idx]["Labels"]
        img = Image.open(img_path)

        if self.transform:
            img = self.transform(img)

        return img, one_hot_label, label


In [4]:
# Load the training dataloader
train_dataloader = torch.load("train_dataloader.pth")

In [5]:
# Initialiaze Computer vision model
model = models.resnet18(pretrained=True)

C:\Non-work\ML\Nepali-liscence-number-recognition\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Non-work\ML\Nepali-liscence-number-recognition\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
# Modify the model to accept grayscale images
model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Get the number of input features in the final fully connected layer
num_features = model.fc.in_features

# Change the number of output feature to match the number of class present in our dataset
model.fc = nn.Linear(num_features, 12)
model.to(device)

ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [7]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

best_accuracy = 0
for epoch in range(15):
    running_loss = 0.0
    epoch_total_correct = 0
    print(f"Epoch: {epoch+1}")
    for i, (inputs, labels, _) in enumerate(train_dataloader):
#         batch_accuracy = 0
        
        optimizer.zero_grad()
        labels = labels.to(device)
        outputs = model(inputs.to(device))
        loss = criterion(outputs, labels.argmax(dim=1))
        loss.backward()
        optimizer.step()

        logits_max = outputs.argmax(dim=1)
        labels_max = labels.argmax(dim=1)

        equals = torch.eq(logits_max, labels_max)
        total_correct_in_batch = int(equals.sum())
        batch_accuracy  = total_correct_in_batch/len(labels_max)
        epoch_total_correct += batch_accuracy


        running_loss += loss.item()
    current_accuracy = epoch_total_correct/len(train_dataloader)
#     print(f"Current accuracy is {current_accuracy}")
    if epoch_total_correct>=best_accuracy:
        best_accuracy = epoch_total_correct
        print("Saving Best model based on accuracy")
        best_model = model
        best_epoch = epoch
        torch.save(best_model, 
            f"trained_model_for_best_accuracy_resnet_on_15_epoch.pt"
        )

    print(f"loss: {running_loss/len(train_dataloader)}\nAccuracy: {current_accuracy}\n***************************************************************************************\n")
    

Epoch: 1
Saving Best model based on accuracy
loss: 1.3763628199933067
Accuracy: 0.569295634920635
***************************************************************************************

Epoch: 2
Saving Best model based on accuracy
loss: 0.3269918927006305
Accuracy: 0.9106150793650793
***************************************************************************************

Epoch: 3
Saving Best model based on accuracy
loss: 0.14535362307455332
Accuracy: 0.9580853174603174
***************************************************************************************

Epoch: 4
Saving Best model based on accuracy
loss: 0.0845642357562033
Accuracy: 0.9766865079365079
***************************************************************************************

Epoch: 5
Saving Best model based on accuracy
loss: 0.05382463521885848
Accuracy: 0.9861111111111112
***************************************************************************************

Epoch: 6
Saving Best model based on accuracy
loss: 0.031675

## prediction

In [10]:
def predict_digit(path):
    transform = T.Compose([
    T.Resize((64,64)),  # Resize image to 28x28
    T.Grayscale(),  # Convert image to grayscale
    T.ToTensor(),  # Convert image to tensor
    T.Normalize((0.5,), (0.5,))  # Normalize image
    ])

    transformed_image = transform(Image.open(path)).unsqueeze(0)
    with torch.no_grad():
        model.eval()  
        output =best_model(transformed_image.to(device))
        index = output.data.cpu().numpy().argmax()
        return index
    

In [12]:
test_data = pd.read_csv("test_data.csv")

In [27]:
len(test_data)

1980

In [29]:
index = 1000
path = test_data.iloc[index]["Images"]
predicted_label = predict_digit(path)
print("predicted label = " + str(predicted_label))
print("Actual label = " + test_data.iloc[index]["Labels"])

predicted label = 5
Actual label = 5


In [22]:
torch.cuda.is_available()

True

In [26]:
next(best_model.parameters()).is_cuda

True

In [24]:
!nvidia-smi

Tue Mar 14 23:16:40 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 516.94       Driver Version: 516.94       CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ... WDDM  | 00000000:01:00.0 Off |                  N/A |
| N/A   40C    P8     4W /  N/A |   1258MiB /  4096MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------